In [1]:
import cellxgene_census

In [2]:
import re
import requests
import time
import yaml
from typing import Optional
from tqdm import tqdm
from unidecode import unidecode

In [3]:
import tiledbsoma

print(tiledbsoma.__version__)

1.17.1


In [4]:
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_dir = notebook_dir.parent
data_dir = notebook_dir / "data/"

In [5]:
# collection_id = "d86517f0-fa7e-4266-b82e-a521350d6d36"
# dataset_id = "f7341911-5051-4bae-8b46-9ddfa880084f"

In [6]:
# cellxgene_census.download_source_h5ad("dbb4e1ed-d820-4e83-981f-88ef7eb55a35", str(data_dir / "single_cell_ref.h5ad"), census_version="latest", progress_bar=True)

In [13]:
with cellxgene_census.open_soma(
    census_version="latest", tiledb_config={"py.init_buffer_bytes": 128 * 1024**2}
) as census:
    cxg_ds = census["census_info"]["datasets"].read().concat().to_pandas()
cxg_ds.head(2)

,soma_joinid,citation,collection_id,collection_name,collection_doi,collection_doi_label,dataset_id,dataset_version_id,dataset_title,dataset_h5ad_path,dataset_total_cell_count
0,0,Publication: https://doi.org/10.1016/j.isci.20...,8e880741-bf9a-4c8e-9227-934204631d2a,High Resolution Slide-seqV2 Spatial Transcript...,10.1016/j.isci.2022.104097,Marshall et al. (2022) iScience,4eb29386-de81-452f-b3c0-e00844e8c7fd,f76861bb-becb-4eb7-82fc-782dc96ccc7f,Spatial transcriptomics in mouse: Puck_191112_05,4eb29386-de81-452f-b3c0-e00844e8c7fd.h5ad,10888
1,1,Publication: https://doi.org/10.1016/j.isci.20...,8e880741-bf9a-4c8e-9227-934204631d2a,High Resolution Slide-seqV2 Spatial Transcript...,10.1016/j.isci.2022.104097,Marshall et al. (2022) iScience,78d59e4a-82eb-4a61-a1dc-da974d7ea54b,7d7ec1b6-6e3f-4aaa-9442-4b22f3424396,Spatial transcriptomics in mouse: Puck_191112_08,78d59e4a-82eb-4a61-a1dc-da974d7ea54b.h5ad,10250


In [11]:
cxg_ds.tail(2)

,soma_joinid,citation,collection_id,collection_name,collection_doi,collection_doi_label,dataset_id,dataset_version_id,dataset_title,dataset_h5ad_path,dataset_total_cell_count
1603,1603,Publication: https://doi.org/10.1038/s41593-02...,1ca90a2d-2943-483d-b678-b809bf464c30,SEA-AD: Seattle Alzheimer’s Disease Brain Cell...,10.1038/s41593-024-01774-5,Gabitto et al. (2024) Nat Neurosci,c2876b1b-06d8-4d96-a56b-5304f815b99a,c32964d2-3339-441f-8e56-7177234c7876,Whole Taxonomy - MTG: Seattle Alzheimer's Dise...,c2876b1b-06d8-4d96-a56b-5304f815b99a.h5ad,1226855
1604,1604,Publication: https://doi.org/10.1038/s41593-02...,1ca90a2d-2943-483d-b678-b809bf464c30,SEA-AD: Seattle Alzheimer’s Disease Brain Cell...,10.1038/s41593-024-01774-5,Gabitto et al. (2024) Nat Neurosci,6f7fd0f1-a2ed-4ff1-80d3-33dde731cbc3,d3427e8c-c55d-4d4e-b15b-1a8774cd3a4b,Whole Taxonomy - DLPFC: Seattle Alzheimer's Di...,6f7fd0f1-a2ed-4ff1-80d3-33dde731cbc3.h5ad,1309414


In [10]:
print(len(cxg_ds))

1605


In [8]:
sc_cxg_ds = cxg_ds[
    cxg_ds["collection_name"].str.contains("single", case=False, na=False)
    & ~cxg_ds["collection_name"].str.contains("spatial", case=False, na=False)
].copy()
print(len(sc_cxg_ds))
sc_cxg_ds.head(2)

309


,soma_joinid,citation,collection_id,collection_name,collection_doi,collection_doi_label,dataset_id,dataset_version_id,dataset_title,dataset_h5ad_path,dataset_total_cell_count
9,9,Publication: https://doi.org/10.1126/sciimmuno...,3a2af25b-2338-4266-aad3-aa8d07473f50,Single-cell analysis of human B cell maturatio...,10.1126/sciimmunol.abe6291,King et al. (2021) Sci. Immunol.,00ff600e-6e2e-4d76-846f-0eec4f0ae417,3fa969b4-25e8-4e0b-b2ec-86cd9ddd2ccb,Human tonsil nonlymphoid cells scRNA,00ff600e-6e2e-4d76-846f-0eec4f0ae417.h5ad,363
23,23,Publication: https://doi.org/10.1038/s41467-02...,bf325905-5e8e-42e3-933d-9a9053e9af80,Single-cell Atlas of common variable immunodef...,10.1038/s41467-022-29450-x,Rodríguez-Ubreva et al. (2022) Nat Commun,a5d95a42-0137-496f-8a60-101e17f263c8,f223d2b0-d878-443b-8bef-f182fe1f22d1,Steady-state B cells - scRNA-seq,a5d95a42-0137-496f-8a60-101e17f263c8.h5ad,1324


In [9]:
# Extract DOIs from the citation strings
doi_re = re.compile(r"https?://doi\.org/\S+", flags=re.IGNORECASE)
doi_links = []
for txt in sc_cxg_ds["citation"]:
    m = doi_re.search(txt)
    doi_links.append(m.group(0) if m else "")
sc_cxg_ds["doi"] = doi_links

In [10]:
HEADERS = {"User-Agent": "AbstractFetcher/1.0 (mailto:you@example.com)"}
CR_ENDPOINT = "https://api.crossref.org/works/"
UP_ENDPOINT = "https://api.unpaywall.org/v2/"


def _crossref_abstract(doi: str) -> Optional[str]:
    r = requests.get(CR_ENDPOINT + doi, headers=HEADERS, timeout=10)
    if r.status_code != 200:
        return None
    raw = r.json().get("message", {}).get("abstract")
    if not raw:
        return None
    return unidecode(
        " ".join(
            segment.strip()
            for segment in raw.replace("<jats:p>", " ").replace("</jats:p>", " ").split()
        )
    )


def _unpaywall_abstract(doi: str, email: str) -> Optional[str]:
    r = requests.get(f"{UP_ENDPOINT}{doi}", params={"email": email}, timeout=10)
    if r.status_code != 200:
        return None
    return r.json().get("oa_locations", [{}])[0].get("abstract")


def _best_abstract(doi: str, email: str) -> Optional[str]:
    for fn in (_crossref_abstract, lambda d: _unpaywall_abstract(d, email)):
        try:
            text = fn(doi)
            if text:
                return text
        except Exception:
            pass
    return None

In [11]:
email = "wenduoc@andrew.cmu.edu"

abstracts = []
for doi in tqdm(sc_cxg_ds["doi"].fillna("").astype(str)):
    doi = doi.strip().lower()
    abstracts.append(_best_abstract(doi, email) if doi else None)
    time.sleep(0.1)
sc_cxg_ds["abstract"] = abstracts

100%|█████████████████████████████████████████| 309/309 [01:15<00:00,  4.11it/s]


In [12]:
sc_cxg_ds["abstract"] = (
    sc_cxg_ds["abstract"]
    .fillna("")
    .str.replace(r"<[^>]*>", "", regex=True)
    .str.replace(r"\s{2,}", " ", regex=True)
    .str.strip()
)

In [13]:
def species_classifier(text: str):
    text = text.lower()
    is_mouse = re.search(r"\b(mouse|mice|murine|mus musculus|c57bl\/6|balb\/c)\b", text)
    if is_mouse:
        return "mouse"
    return "human"


sc_cxg_ds["species"] = sc_cxg_ds.apply(
    lambda row: species_classifier(
        f"{row['collection_name']}\n{row['dataset_title']}\n{row['abstract']}"
    ),
    axis=1,
)

In [14]:
num_mouse = sum(sc_cxg_ds["species"] == "mouse")
print(f"mouse: {num_mouse} / human: {len(sc_cxg_ds) - num_mouse}")

mouse: 93 / human: 216


In [15]:
tissue_patterns = {
    "brain": r"\b(cortex|hippocampus|cerebellum|hypothalamus|medulla|midbrain|brainstem|substantia nigra|nigra|white matter|snpc|dopaminergic|dopamine neuron|oligodendrocyte|microglia|glioblastoma|brain|cns|neuronal)\b",
    "lung": r"\b(alveolus|alveolar|bronch(?:us|i(?:ole)?)|trachea|airway|lung|pulmonary|respiratory epithelium)\b",
    "heart": r"\b(myocard(?:ium)?|ventricle|atrium|atrial|cardiac|heart|cardiomyocyte)\b",
    "liver": r"\b(hepatocyte|hepatic|liver)\b",
    "kidney": r"\b(nephron|glomerulus|renal|kidney|ccrcc)\b",
    "pancreas": r"\b(islet|beta[- ]cell|acinar cell|pancreatic|pancreas)\b",
    "intestine": r"\b(duodenum|jejunum|ileum|colon|enterocyte|intestinal|intestine|gut)\b",
    "spleen": r"\b(splenic|spleen)\b",
    "lymph_node": r"\b(lymph node|lymphatic vasculature|lymphatic endothelial|lymphatic vessel|inguinal lymph node)\b",
    "breast": r"\b(breast|mammary gland|mammary tissue|mammary)\b",
    "uterus": r"\b(uterus|uterine|endometrium|myometrium|decidua|pregnant uterus)\b",
    "placenta": r"\b(placenta|placental)\b",
    "blood": r"\b(pbmc|peripheral blood|bone marrow|hematopoietic|lymphocyte|b[- ]cell|t[- ]cell|blood)\b",
    "skin": r"\b(keratinocyte|epidermis|dermis|skin|cutaneous)\b",
    "muscle": r"\b(skeletal muscle|myofiber|myocyte|muscle)\b",
    "adipose": r"\b(brown adipose|white adipose|adipocyte|fat pad|adipose)\b",
    "eye": r"\b(retina|cornea|ocular|optic nerve|eye)\b",
    "testis": r"\b(testis|testes|testicular)\b",
    "ovary": r"\b(ovary|ovarian)\b",
    "thyroid": r"\b(thyroid)\b",
    "oral": r"\b(oral|buccal|gingiv\w+|salivary gland|parotid|sublingual|submandibular|minor salivary gland|acinar cell|dental pulp|dental|tooth|teeth|maxillary|mandibular)\b",
    "unknown": r"",
}


tissue_patterns = {tissue: re.compile(pat, flags=re.I) for tissue, pat in tissue_patterns.items()}


def tissue_classifier(text: str):
    text = text.lower()
    for tissue, pattern in tissue_patterns.items():
        if pattern.search(text):
            return tissue
    return "unknown"


sc_cxg_ds["tissue"] = sc_cxg_ds.apply(
    lambda row: tissue_classifier(
        f"{row['collection_name']}\n{row['dataset_title']}\n{row['abstract']}"
    ),
    axis=1,
)

In [16]:
sum(sc_cxg_ds["tissue"] == "unknown")

11

In [17]:
condition_patterns = {
    "covid19": r"\b(covid[- ]?19|sars[- ]cov[- ]?2|2019[- ]nco?v)\b",
    "cancer": r"\b(cancer|tumou?r|carcinoma|sarcoma|leukemia|lymphoma|glioblastoma|melanoma|myeloma|neoplasm|malignan\w+|adenoma|oncolog\w+)\b",
    "autoimmune": r"\b(autoimmune|sjogren['’]s?|rheumatoid arthritis|multiple sclerosis|lupus|systemic lupus erythematosus|sle|crohn['’]s?|ulcerative colitis)\b",
    "infectious_disease": r"\b(infection|infectious|viral|bacterial|influenza|hiv|aids|ebola|zika|dengue|sepsis|pathogen)\b",
    "neurodegenerative": r"\b(alzheimer['’]s?|parkinson['’]s?|amyotrophic lateral sclerosis|als|huntington['’]s?|neurodegenerat\w+|dementia|tauopathy)\b",
    "immunodeficiency": r"\b(cvid|common variable immunodeficiency|immunodeficiency|immunocompromised)\b",
    "metabolic": r"\b(diabetes|obesity|metabolic syndrome|nash|nafld|non[- ]alcoholic (fatty liver|steatohepatitis))\b",
    "cardiovascular": r"\b(cardiovascular|heart failure|coronary artery|atherosclerosis|myocardial infarction|stroke|ischemia)\b",
    "healthy": r"\b(healthy (donor|control)s?|normal (donor|control)s?|non[- ]diseased|non[- ]tumou?r)\b",
}

condition_patterns = {
    disease: re.compile(pat, flags=re.I) for disease, pat in condition_patterns.items()
}


def condition_classifier(text: str):
    text = text.lower()
    for condition, pattern in condition_patterns.items():
        if pattern.search(text):
            return condition
    return "healthy"


sc_cxg_ds["disease"] = sc_cxg_ds.apply(
    lambda row: condition_classifier(
        f"{row['collection_name']}\n{row['dataset_title']}\n{row['abstract']}"
    ),
    axis=1,
)

In [18]:
sc_cxg_ds.to_csv(data_dir / "sc_cxg_ds.tsv", sep="\t")
with (data_dir / "sc_cxg_params.yaml").open("w") as f:
    yaml.safe_dump(
        {
            "species": ["human", "mouse"],
            "tissue": list(tissue_patterns.keys()),
            "disease": list(condition_patterns.keys()),
        },
        f,
        sort_keys=False,
    )